# 🚀 Estratégia de Modelagem: Pivot para Propensão à Recompra (Propensity Scoring)

## 1. Contexto e O Desafio dos "97%"
Na etapa de *Exploratory Data Analysis (EDA)*, identificamos um desbalanceamento extremo na base de dados da Olist:
* **~97%** dos clientes não realizam uma segunda compra (Churn Natural).
* Apenas **~3%** retornam.

Diante desse cenário, uma abordagem tradicional de **Classificação Binária (Hard Class)** — onde o modelo decide apenas "Sim" (1) ou "Não" (0) — torna-se ineficaz. Um modelo que previsse "Não" para 100% da base teria 97% de acurácia, mas valor de negócio nulo.

## 2. A Nova Abordagem: Propensity Scoring
Ao invés de tentar classificar deterministicamente se o cliente volta ou não, nosso objetivo muda para calcular a **Probabilidade de Retorno** (`predict_proba`).

O modelo atuará como um algoritmo de **Ranking (Ordenação)**:
* O objetivo não é acertar o rótulo perfeito.
* O objetivo é garantir que clientes com alta probabilidade tenham scores maiores que clientes com baixa probabilidade.

### Metodologia de Treino vs. Inferência
* **Treino (Supervisionado):** Continuaremos usando o alvo binário (`target = 1` se recomprou em 90 dias, `0` caso contrário). O modelo aprenderá a minimizar o erro (Log Loss) entre a previsão e esse fato histórico.
* **Inferência (Aplicação):** Ignoraremos a classe predita (`predict`) e utilizaremos exclusivamente a probabilidade da classe positiva (`predict_proba`), gerando um score contínuo de 0.0 a 1.0 para cada cliente.

## 3. Aplicação de Negócio: Segmentation & Uplift Proxy
Com o score de propensão em mãos, a estratégia de marketing deixa de ser binária ("Mandar cupom" vs "Não mandar") e passa a ser segmentada baseada na "Zona de Persuasão":

1.  **Causas Perdidas (Probabilidade Baixa | ex: < 10%):**
    * Clientes frios. O custo de marketing supera o retorno esperado.
    * *Ação:* Não intervir.
2.  **Os Persuadíveis (Probabilidade Média | ex: 10% - 60%):**
    * Clientes "em cima do muro". É onde o Marketing tem maior ROI (Retorno sobre Investimento).
    * *Ação:* **Foco total de campanhas e incentivos (Cupons/Push).**
3.  **Os Garantidos (Probabilidade Alta | ex: > 80%):**
    * Clientes fiéis que provavelmente comprariam organicamente.
    * *Ação:* Não oferecer descontos agressivos para não queimar margem desnecessariamente.

## 4. Métricas de Sucesso
Como não estamos focando em Acurácia, o sucesso do protótipo será medido pela capacidade de ordenação do modelo:
* **ROC-AUC:** Capacidade global de distinguir classes.
* **Precision-Recall Curve:** Foco na classe minoritária (os 3% que compram).
* **Lift @ Decile (Top 10%):** Quantas vezes o nosso modelo é melhor em encontrar compradores no top 10% da base comparado a uma escolha aleatória?